In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os

from pydantic import BaseModel

from doc_intelligence import PDFProcessor

In [3]:
import sys

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from utils import show_pdf_with_bboxes

PDF_URL = "https://example-files.online-convert.com/document/pdf/example.pdf"
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output")
os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
class License(BaseModel):
    license_name: str

In [5]:
processor = PDFProcessor(provider="ollama")

In [6]:
response = processor.extract(
    uri=PDF_URL,
    response_format=License,
    include_citations=True,
    extraction_mode="single_pass",
    page_numbers=[0, 1],
    llm_config={
        "model": "qwen3.5:9b",
        "think": False,
    },
)

2026-03-13 02:42:32.226 | INFO     | doc_intelligence.pdf.processor:_parse:74 - Document parsed successfully
2026-03-13 02:42:32.226 | DEBUG    | doc_intelligence.pdf.extractor:_run_single_pass:109 - DigitalPDFExtractor: extract: json_instance_schema: {
    "license_name": {
        "value": <string>,
        "citations": [{"page": <integer>, "lines": [<integer>]}]
    }
}
2026-03-13 02:42:32.227 | DEBUG    | doc_intelligence.pdf.extractor:_run_single_pass:113 - DigitalPDFExtractor: extract: content_text: <page number=0>
0: PDF test file
1: Purpose: Provide example of this file type
2: Document file type: PDF
3: Version: 1.0
4: Remark:
5: Example content:
6: The names "John Doe" for males, "Jane Doe" or "Jane
7: Roe" for females, or "Jonnie Doe" and "Janie Doe" for
8: children, or just "Doe" non-gender-specifically are used as
9: placeholder names for a party whose true identity is un-
10: known or must be withheld in a legal action, case, or dis-
11: cussion. The names are also used t

In [7]:
response

ExtractionResult(data=License(license_name='Attribution-ShareAlike 3.0 Unported'), metadata={'license_name': {'value': 'Attribution-ShareAlike 3.0 Unported', 'citations': [{'page': 0, 'bboxes': [{'x0': 0.20106913928643427, 'top': 0.8587326111744586, 'x1': 0.5648947389639185, 'bottom': 0.8718454960091222}]}]}})

In [8]:
assert response.metadata is not None
show_pdf_with_bboxes(
    PDF_URL,
    response.metadata["license_name"]["citations"][0],
    out_file=os.path.join(OUT_DIR, "single_pass_bbox.png"),
)

  -> saved to /Users/zeel/Public/ms/open_source/document_ai/notebooks/pdf/output/single_pass_bbox.png
